# CDH Metadata Assistant

Turn any geospatial dataset into a **valid Climate Data Hub YAML metadata record** by answering a
short questionnaire. A Gemini agent loads the [`cdh-metadata`](https://github.com/CGIAR-Climate-Data-Hub/skills)
skill, inspects your file for everything it can measure, and asks you only for what a file cannot tell
it — what the data *means*, who owns it, and how it should be cited.

---

### How to use this notebook

1. **Runtime ▸ Run all.** The setup cells install everything and load the skill.
2. When the chat box appears, **answer each question** and press *Send* (or Enter).
   If the agent proposes a value, press **Enter on an empty box to accept it**.
3. When it says *Finished*, run the last cell to **print and download your `.yaml`**.

Most cells are collapsed to a single title bar — click ▸ on any of them to read the code.
The two cells you may actually want to change are marked **✏️ EDIT ME**.

### Before you start

| Requirement | How |
|---|---|
| Gemini API key | Create one at [aistudio.google.com](https://aistudio.google.com/apikey), then add it in Colab under 🔑 **Secrets** as `GOOGLE_API_KEY` and enable notebook access. |
| A dataset | A local path, a public URL, or just the name of a well-known public dataset (e.g. *CHIRTS-daily*). |

> **Free-tier keys work.** The agent batches its questions — it asks a dozen at once, you answer
> them one at a time locally, and the notebook spends a single API call per batch. A full record
> normally takes 4–6 calls. If you do hit the per-minute limit, the notebook waits and resumes by
> itself; no need to restart.

> **What the agent will not invent.** Licence, contact/licensor, citation, DOI and data URLs always come
> from you. Titles, descriptions and keywords may be *proposed* for well-known public datasets — read
> them before accepting; they become the catalogue's public record of your data.
>
> **Every URL gets resolved before it reaches the record.** Links you supply and links the agent
> drafts are both checked live from this notebook — a drafted link that does not resolve is dropped,
> and one of yours that fails comes back to you with its status code.
>
> **And the finished record is validated.** The notebook checks it against the published CDH JSON
> schema before the agent may call it done, and again when you export it. *Valid* here means a
> schema said so, not that the agent believes so.

---
## 1 · Setup

Installs the SDK, downloads the CDH skill, and connects to the Gemini API.

In [ ]:
#@title Install the Gemini SDK { display-mode: "form" }
%pip install -U -q "google-genai>=2.9.0" "jsonschema>=4.18" pyyaml

In [ ]:
#@title Download the `cdh-metadata` skill { display-mode: "form" }
import pathlib

!npx -y skills add https://github.com/cgiar-climate-data-hub/skills --skill cdh-metadata --agent antigravity -y

SKILL_ROOT = pathlib.Path("/content/.agents/skills/")
SKILL_NAME = "cdh-metadata"
MOUNT      = "/content/.agents/skills"        # where the skill is mounted inside the agent sandbox
SKILL_DIR  = f"{MOUNT}/{SKILL_NAME}"

assert (SKILL_ROOT / SKILL_NAME / "SKILL.md").exists(), "Skill download failed - re-run this cell."
print(f"Skill ready: {SKILL_ROOT / SKILL_NAME}")

In [ ]:
#@title Connect to the Gemini API { display-mode: "form" }
SECRET_NAME = "GOOGLE_API_KEY" #@param {type:"string"}

import uuid, json
from google import genai
from google.colab import userdata
from IPython.display import display, Markdown
import ipywidgets as widgets

client = genai.Client(api_key=userdata.get(SECRET_NAME))
AGENT  = "antigravity-preview-05-2026"
print("Connected.")

---
## 2 · Chat plumbing

Nothing here needs changing. Three tools run **in this notebook** rather than in the agent's sandbox,
because the sandbox has neither a keyboard nor open internet access:

- **`ask_user`** — the agent poses one question; the API pauses with `status = "requires_action"`
  and this code turns that pause into a text box.
- **`check_url`** — the agent hands over a URL; the notebook resolves it and reports back
  ✅ live, ↪️ redirected, 🔒 access-gated, ❌ dead or ⚠️ unreachable.

- **`validate_record`** — the agent hands over the finished YAML; the notebook parses it, fetches
  the CDH schema the record itself declares, and returns ✅ valid or the exact list of schema errors.

`check_url` is what keeps dead links out of the catalogue. Every URL destined for the record gets
resolved here before the record is written, including URLs the agent drafted from memory.
`validate_record` is what keeps the word *valid* honest: the agent may not call the record finished
until this function says it validates, and it is told to fix and re-check until it does.

**Why answering is free.** Both tools are *batched*: the agent sends a dozen questions in one turn,
you answer them one at a time in the box below, and only when the batch is done does the notebook
spend a single API call sending every answer back. Questions cost nothing — **turns** cost. The cell
also backs off and retries automatically when the free tier returns a `429`.

In [ ]:
#@title Helper functions - chat UI, `ask_user` and `check_url` { display-mode: "form" }
import urllib.request, urllib.error, urllib.parse, socket, time, re
import yaml
from jsonschema import Draft202012Validator

UA = {"User-Agent": "Mozilla/5.0 (compatible; CDH-metadata-bot/1.0)"}
MAX_RETRIES = 4

# --- tool 1: interview the author --------------------------------------------
ask_user_tool = {
    "type": "function",
    "name": "ask_user",
    "description": (
        "Ask the dataset author for a BATCH of fields in a single call, and receive every answer "
        "together. Use this for the fields the cdh-metadata skill marks as 'always ask' (id, "
        "title, description, license, contact, citation/DOI, data URLs, cdh.domain, keywords, "
        "and the meaning/units/caveats of each variable).\n"
        "Put EVERY field you currently need into `questions` - typically 10-15 at once, and on "
        "your first call every 'always ask' field you cannot derive. The client presents them to "
        "the user one at a time and returns all the answers in one response, so a batch of 15 "
        "costs the same as a batch of 1. Do NOT call this with a single question and wait: that "
        "burns the user's whole token quota and is the most common way this workflow fails. "
        "Hold a field back for a later call only when its wording genuinely cannot be written "
        "until an earlier answer arrives."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "questions": {
                "type": "array",
                "minItems": 1,
                "description": "Every field you need right now. Longer is better.",
                "items": {
                    "type": "object",
                    "properties": {
                        "field":      {"type": "string", "description": "CDH field being filled, e.g. 'license'."},
                        "question":   {"type": "string", "description": "The question, in plain language."},
                        "suggestion": {"type": "string", "description": "Optional draft value the user accepts with Enter."},
                    },
                    "required": ["field", "question"],
                },
            },
        },
        "required": ["questions"],
    },
}

# --- tool 2: resolve a URL from the notebook, not the sandbox ------------------
check_url_tool = {
    "type": "function",
    "name": "check_url",
    "description": (
        "Resolve one URL and report whether it is live. Runs on the user's machine, which has "
        "open internet access - your own sandbox does not, so this is the ONLY way to verify a "
        "link. Call it for every URL that will appear in the record, including ones you drafted "
        "yourself, before presenting the record as finished. For a DOI, pass "
        "https://doi.org/<doi>.\n"
        "Emit one call per URL, all in the SAME turn - the client resolves the whole batch and "
        "returns every result together."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "url":   {"type": "string", "description": "Absolute URL to resolve."},
            "field": {"type": "string", "description": "Which record field this URL is for."},
        },
        "required": ["url", "field"],
    },
}

# --- tool 3: validate the finished record against the published CDH schema -----
validate_record_tool = {
    "type": "function",
    "name": "validate_record",
    "description": (
        "Validate a complete CDH YAML record against the published CDH JSON schema. Runs on the "
        "user's machine, which can reach the schema host - your sandbox cannot, so this is the "
        "ONLY schema check available to you. Call it with the full record text before you tell "
        "the user the record is finished, then fix every error it reports and call it again, "
        "until it returns state 'valid'. Never describe a record as valid unless this function "
        "said so in this session."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "record_yaml": {
                "type": "string",
                "description": "The complete YAML record, exactly as it will be written to disk.",
            },
        },
        "required": ["record_yaml"],
    },
}

TOOLS = [{"type": "code_execution"}, ask_user_tool, check_url_tool, validate_record_tool]


def check_url(url, field="", **_):
    """HEAD the URL, falling back to a ranged GET when the host refuses HEAD."""
    if url.startswith("s3://"):
        return {"state": "unchecked", "url": url,
                "note": "s3:// cannot be resolved over HTTP - check the HTTPS endpoint instead"}
    if not url.startswith(("http://", "https://")):
        return {"state": "invalid", "url": url, "note": "not an absolute http(s) URL"}

    def _open(method, extra=None):
        req = urllib.request.Request(url, method=method, headers={**UA, **(extra or {})})
        return urllib.request.urlopen(req, timeout=20)

    try:
        try:
            r = _open("HEAD")
        except urllib.error.HTTPError as e:
            if e.code not in (403, 405, 501):
                raise
            r = _open("GET", {"Range": "bytes=0-0"})   # some hosts refuse HEAD outright
        with r:
            final = r.url
            moved = final.rstrip("/") != url.rstrip("/")
            if moved and url.split("/")[2] in ("doi.org", "dx.doi.org"):
                # a DOI that resolves is valid; the record keeps the bare DOI, not the
                # landing page (which often carries cookie/tracking query params)
                return {"state": "ok", "url": url, "status": r.status,
                        "note": f"DOI resolves to {final.split('?')[0]} - keep the bare DOI"}
            return {"state": "moved" if moved else "ok", "url": url, "status": r.status,
                    "final_url": final.split("?")[0] if moved else None,
                    "content_type": r.headers.get("Content-Type")}
    except urllib.error.HTTPError as e:
        state = "gated" if e.code in (401, 403) else "dead"
        return {"state": state, "url": url, "status": e.code, "note": str(e.reason)}
    except (urllib.error.URLError, socket.timeout, ValueError, OSError) as e:
        return {"state": "unreachable", "url": url, "status": None, "note": str(e)}


SCHEMA_HOST    = "cgiar-climate-data-hub.github.io"
DEFAULT_SCHEMA = ("https://cgiar-climate-data-hub.github.io/cdh-metadata-standard/"
                  "v0.3.0/schemas/profiles/cdh.schema.json")
_schema_cache  = {}


def _schema_url(record):
    """The *bundled* copy of whatever profile the record declares: it inlines every
    $ref, so validating costs one fetch and needs no reference resolver."""
    url = record.get("$schema") or DEFAULT_SCHEMA
    if url.endswith("/cdh.schema.json"):
        url = url[: -len("cdh.schema.json")] + "cdh.schema.bundled.json"
    return url


def validate_record(record_yaml, **_):
    """Parse the record, fetch the schema it declares, and return every violation."""
    try:
        record = yaml.safe_load(record_yaml)
    except yaml.YAMLError as exc:
        return {"state": "invalid_yaml", "note": str(exc)[:500]}
    if not isinstance(record, dict):
        return {"state": "invalid_yaml", "note": "the record must be a YAML mapping"}

    url = _schema_url(record)
    if urllib.parse.urlparse(url).hostname != SCHEMA_HOST:
        return {"state": "unavailable", "note": f"refusing to fetch a schema from {url}"}
    if url not in _schema_cache:
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=30) as r:
                _schema_cache[url] = json.loads(r.read())
        except Exception as exc:
            return {"state": "unavailable", "checked_against": url,
                    "note": f"could not fetch the schema: {exc}"}

    errors = sorted(Draft202012Validator(_schema_cache[url]).iter_errors(record),
                    key=lambda e: list(e.absolute_path))
    # A failing subschema drops its annotations, so the root `unevaluatedProperties`
    # error fires as a side effect of any other error - and when it does it names
    # every field in the record, mandatory ones included. Pure noise, unless it is
    # the only error, in which case it names the genuinely unknown field.
    real  = [e for e in errors if e.validator != "unevaluatedProperties" or e.absolute_path]
    shown = real or errors
    if not shown:
        return {"state": "valid", "checked_against": url}
    return {"state": "invalid", "checked_against": url, "error_count": len(shown),
            "errors": [{"path": "/".join(str(p) for p in e.absolute_path) or "(root)",
                        "message": e.message[:300]} for e in shown[:25]]}


ICON = {"ok": "✅", "moved": "↪️", "gated": "🔒", "dead": "❌",
        "unreachable": "⚠️", "unchecked": "➖", "invalid": "❌"}


def run_local(call):
    """Execute a non-interactive tool call here in the notebook."""
    args = call_args(call)
    if call.name == "check_url":
        res = check_url(**args)
        line = (f"{ICON.get(res['state'], '?')} `{res['state']}` "
                f"{res.get('status') or ''} {res['url']}")
        if res.get("final_url"):
            line += f" → {res['final_url']}"
        if res.get("note"):
            line += f" — *{res['note']}*"
        with out:
            display(Markdown(line))
        return res
    if call.name == "validate_record":
        res  = validate_record(**args)
        mark = {"valid": "✅", "invalid": "❌",
                "invalid_yaml": "❌", "unavailable": "⚠️"}.get(res["state"], "?")
        line = f"{mark} `{res['state']}`"
        if res["state"] == "invalid":
            line += f" — {res['error_count']} schema error(s)\n" + "\n".join(
                f"- `{e['path']}` — {e['message']}" for e in res["errors"])
        elif res.get("note"):
            line += f" — *{res['note']}*"
        else:
            line += " — the record validates against the CDH schema"
        with out:
            display(Markdown(line))
        return res
    return {"error": f"no local handler for {call.name}"}


# --- widgets ------------------------------------------------------------------
out  = widgets.Output()
box  = widgets.Text(layout=widgets.Layout(width="80%"))
send = widgets.Button(description="Send", button_style="primary")
ui   = widgets.HBox([box, send])

state = {"result": None, "env": None, "queue": [], "answers": [],
         "subq": None, "collected": {}, "batch": 0, "turns": 0}


def api_call(**kwargs):
    """interactions.create with backoff, so a free-tier 429 pauses instead of crashing."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            res = client.interactions.create(**kwargs)
            state["turns"] += 1
            return res
        except Exception as exc:
            text = str(exc)
            if "429" not in text and "too_many_requests" not in text.lower():
                raise
            found = re.search(r"retry in ([\d.]+)s", text)
            wait = float(found.group(1)) + 2 if found else 20 * attempt
            with out:
                print(f"rate limited - waiting {wait:.0f}s "
                      f"(attempt {attempt}/{MAX_RETRIES}; the notebook will resume on its own)")
            time.sleep(wait)
    raise RuntimeError(
        "Still rate limited after several retries. Free-tier input tokens reset each minute - "
        "wait a little and re-run the cell, or use a paid key."
    )


def show(obj, prefix="**Agent**\n\n"):
    """Render an interaction's text as Markdown (silent when the turn was tool calls only)."""
    text = obj if isinstance(obj, str) else getattr(obj, "output_text", None)
    if text:
        display(Markdown(prefix + text))


def pending_calls(res):
    """Tool calls the agent is still waiting on."""
    done = {s.call_id for s in res.steps if s.type == "function_result"}
    return [s for s in res.steps if s.type == "function_call" and s.id not in done]


def call_args(call):
    return call.arguments if isinstance(call.arguments, dict) else json.loads(call.arguments)


def reply(call, payload):
    state["answers"].append({
        "type": "function_result", "name": call.name,
        "call_id": call.id, "result": payload,
    })


def end(msg):
    box.disabled = send.disabled = True
    with out:
        display(Markdown(f"### {msg}"))


def questions_of(call):
    """The question list inside an ask_user call (tolerates the old single-question shape)."""
    args = call_args(call)
    items = args.get("questions")
    if isinstance(items, list) and items:
        return [q for q in items if isinstance(q, dict) and q.get("question")]
    if args.get("question"):
        return [{"field": args.get("field", "value"), "question": args["question"],
                 "suggestion": args.get("suggestion")}]
    return []


def pose(q):
    """Show one question from the current batch and hand control to the text box."""
    done = state["batch"] - len(state["subq"]) + 1
    counter = f"`{done}/{state['batch']}` " if state["batch"] > 1 else ""
    question = f"{counter}**{q.get('field', 'question')}** - {q['question']}"
    if q.get("suggestion"):
        question += (f"\n\n> *Proposed:* {q['suggestion']}\n>\n"
                     "> *Press Enter on an empty box to accept, or type a replacement.*")
    with out:
        display(Markdown(question))
    box.placeholder = "Enter to accept" if q.get("suggestion") else "Your answer..."
    box.disabled = send.disabled = False
    box.value = ""


def advance_turn():
    """Send the batch of tool results back and pick up whatever comes next."""
    box.disabled = send.disabled = True
    with out:
        print(f"thinking... (turn {state['turns'] + 1}, sending {len(state['answers'])} result(s))")
    state["result"] = api_call(
        agent=cdh_agent.id,
        previous_interaction_id=state["result"].id,
        environment=state["env"],
        input=state["answers"],
        tools=TOOLS,
    )
    state["answers"] = []
    with out:
        show(state["result"])
    ask_next()


def ask_next():
    """Drive the queue: resolve URLs silently, pause only for questions."""
    while True:
        while state["queue"] and state["queue"][0].name != "ask_user":
            call = state["queue"].pop(0)
            reply(call, run_local(call))

        if state["queue"]:
            call = state["queue"][0]
            if state["subq"] is None:                  # opening a fresh batch
                state["subq"] = questions_of(call)
                state["collected"] = {}
                state["batch"] = len(state["subq"])
                if state["batch"] > 1:
                    with out:
                        display(Markdown(
                            f"*{state['batch']} questions in this batch - answering all of them "
                            "costs a single API turn.*"))
                elif state["batch"] == 1:
                    with out:
                        display(Markdown(
                            "*The agent sent only one question this turn - that is slow and "
                            "quota-hungry. If it keeps doing this, re-run the agent cell.*"))

            if state["subq"]:                          # more questions in this batch
                return pose(state["subq"][0])

            state["queue"].pop(0)                      # batch complete -> one result
            reply(call, {"answers": state["collected"]})
            state["subq"] = None
            continue

        if state["answers"]:               # tool results ready to send back
            return advance_turn()

        res = state["result"]              # nothing pending - is the agent done?
        if res.status != "requires_action":
            return end(f"Finished in {state['turns']} API turns - scroll up for the record.")
        calls = pending_calls(res)
        if not calls:
            return end(f"Finished in {state['turns']} API turns - nothing left to answer.")
        state["queue"] = calls


def submit(_):
    """Record one answer from the current batch and continue."""
    if not state["subq"] or box.disabled:
        return
    msg, box.value = box.value.strip(), ""
    q = state["subq"].pop(0)
    answer = msg or q.get("suggestion") or ""
    with out:
        display(Markdown(f"**You** - {answer}"))
    state["collected"][q.get("field", f"q{len(state['collected'])}")] = answer
    ask_next()


send.on_click(submit)
box.on_submit(submit)   # Enter key (deprecated in ipywidgets 8 - harmless warning)


def start(first_input):
    """Open the interview."""
    out.clear_output()
    state.update(result=None, env=None, queue=[], answers=[],
                 subq=None, collected={}, batch=0, turns=0)
    state["result"] = api_call(
        agent=cdh_agent.id, input=first_input, environment="remote", tools=TOOLS,
    )
    state["env"] = state["result"].environment_id
    display(out, ui)
    with out:
        show(state["result"])
    ask_next()


print("Helpers loaded.")

---
## 3 · Agent instructions &nbsp; ✏️ **EDIT ME**

This is the agent's standing brief: who it is, where the skill lives, and — most importantly —
**which fields it may draft for you and which it must never guess**. Widen the *PROPOSE* list to be
asked fewer questions; move a field into *NEVER GUESS* to be asked about it explicitly every time.

In [ ]:
SYSTEM_INSTRUCTION = (
    "You are a geospatial data curator for the CGIAR Climate Data Hub (CDH). "
    "You produce valid CDH YAML metadata records (standard v0.3.0) for raster, "
    "vector, NetCDF and Zarr datasets.\n"

    f"The `cdh-metadata` skill is mounted at {SKILL_DIR}/SKILL.md. "
    "Read it in full before doing anything else and follow its stages in order: "
    "inspect the file, ask the user for the fields that cannot be derived, then "
    "write and validate the YAML.\n"
    f"Consult {SKILL_DIR}/references/cdh-annotated-template.md "
    "whenever you need a field's exact shape or the list of optional fields.\n"

    "\nYou have no chat channel with the user. The ONLY way to ask the dataset author "
    "anything is to call the `ask_user` function - one call carrying every field you "
    "currently need. A question written as plain message text will never be answered, so "
    "never end a message with a question.\n"

    "\nBATCH YOUR QUESTIONS. This matters more than anything else here: the user is on a "
    "metered quota and every turn resends the whole conversation, so a turn that asks one "
    "question costs as much as a turn that asks fifteen. `ask_user` takes a LIST - fill it. "
    "Your FIRST ask_user call must contain every 'always ask' field you cannot derive from the "
    "file: id, title, description, license, contact, citation or DOI, data locations, "
    "cdh.domain, keywords, and the units and meaning of every variable. Never send a "
    "one-question list and wait - that is the single most common way this workflow fails. "
    "Likewise emit every check_url call you need in the same turn. Aim to finish the whole "
    "record in under six turns.\n"

    "\nBe frugal with reading, too. Read SKILL.md once. Do NOT read "
    "references/cdh-annotated-template.md unless a specific field is genuinely unclear - it is "
    "large, and once read it is resent on every later turn. Never re-read a file you have "
    "already read.\n"

    "\nYour sandbox has almost no outbound network access, so you cannot resolve URLs yourself. "
    "The `check_url` function runs on the user's machine and is the only way to verify a link. "
    "Call it for EVERY url that will appear in the record - data locations, citation url, DOI "
    "(as https://doi.org/<doi>), series url, processing source, contact url - and for the "
    "$schema and extensions[] URLs before you write the file. Never describe a link as working "
    "unless check_url returned ok or moved for it in this session.\n"
    "On failure: a URL you drafted yourself is dropped from the record, no exceptions; a URL the "
    "user gave you is raised back to them once with ask_user, quoting the status code; a dead "
    "$schema or extension URL stops the run. On 'moved', use the final URL and say that you "
    "followed a redirect.\n"

    "\nTwo kinds of fields:\n"
    "- PROPOSE - title, description, keywords, cdh.domain, series, variable units and variable "
    "descriptions, and spatial/temporal extent for well-known public datasets. If you recognise "
    "the dataset, draft a value and pass it as `suggestion` on the ask_user call, phrasing the "
    "question as a confirmation ('Accept, or edit:'). Say in the question that the value is your "
    "proposal, not something read from the file.\n"
    "- NEVER GUESS - license, contact/licensor, citation, DOI, data URLs, and anything about an "
    "unpublished or local dataset. Ask these with no suggestion. If you are not confident the "
    "dataset is the well-known public one, treat the field as NEVER GUESS.\n"

    "\nBefore writing the final YAML, echo back every value you proposed and ask the author to "
    "confirm the set.\n"

    "\nThen VALIDATE. Call `validate_record` with the complete record text - it runs on the "
    "user's machine and is the only schema check you have, since your sandbox cannot reach "
    "the schema host. Fix every error it returns and call it again, until it answers state "
    "'valid'. Do not present the record as valid, or as finished, before that - a record you "
    "have not put through `validate_record` is a draft. If it answers 'unavailable', say "
    "plainly in your summary that the record is UNVALIDATED rather than implying it passed."
)

print(SYSTEM_INSTRUCTION)

---
## 4 · Create the agent

Ships the skill files into the agent's sandbox as inline sources and registers the agent.
Re-run this cell after any edit to the instructions above.

In [ ]:
#@title Build the sandbox and create the agent { display-mode: "form" }

def skill_sources(root, skill, mount=MOUNT):
    """Turn the locally downloaded skill folder into inline environment sources."""
    src_dir, sources = root / skill, []
    for p in sorted(src_dir.rglob("*")):
        if not p.is_file() or "evals" in p.parts:      # evals aren't needed at runtime
            continue
        sources.append({
            "type": "inline",
            "target": f"{mount}/{skill}/{p.relative_to(src_dir).as_posix()}",
            "content": p.read_text(encoding="utf-8"),
        })
    return sources


SOURCES = skill_sources(SKILL_ROOT, SKILL_NAME)

cdh_agent = client.agents.create(
    id=f"cdh-metadata-agent-{uuid.uuid4().hex[:8]}",   # fresh id so re-runs never collide
    base_agent=AGENT,
    system_instruction=SYSTEM_INSTRUCTION,
    base_environment={
        "type": "remote",
        "sources": SOURCES,
        "network": {"allowlist": [
            {"domain": "cgiar-climate-data-hub.github.io"},   # CDH JSON schema
            {"domain": "raw.githubusercontent.com"},          # authoring guide
            {"domain": "pypi.org"},                           # rasterio / xarray
            {"domain": "files.pythonhosted.org"},
        ]},
    },
)

print(f"Agent ready: {cdh_agent.id}")
for s in SOURCES:
    print("  mounted", s["target"])

---
## 5 · Describe your dataset &nbsp; ✏️ **EDIT ME**

Say what you want documented. Be as specific as you can — a **file path** is best, because it lets the
agent measure the bounding box, CRS, resolution, dtypes and nodata itself instead of asking you.

| You have | Write something like |
|---|---|
| A file in the runtime | `Create a CDH metadata record for /content/data/chirts_tmax_2024.tif` |
| A public dataset | `Create a CDH metadata record for CHIRTS-daily` |
| A public URL | `Create a CDH metadata record for https://.../tmax_2024.nc` |

Then answer the questions in the box that appears below.

In [ ]:
PROMPT = "Create a CDH metadata record for CHIRTS-daily" #@param {type:"string"}

start(PROMPT)

---
## 6 · Export the record

Run this once the chat above says **Finished**. It prints the YAML and downloads it as `cdh_record.yaml`.

In [ ]:
#@title Print and download the YAML { display-mode: "form" }
import re

final = api_call(
    agent=cdh_agent.id,
    previous_interaction_id=state["result"].id,
    environment=state["env"],
    input="Print the final CDH YAML record in a single fenced yaml block. No commentary.",
    tools=TOOLS,
)
display(Markdown(final.output_text or "_no output_"))

match = re.search(r"```(?:yaml)?\s*\n(.*?)```", final.output_text or "", re.S)
if match:
    record = match.group(1).strip() + "\n"

    # Validate here too. The agent was told to, but this is the copy you keep.
    verdict = validate_record(record)
    if verdict["state"] == "valid":
        display(Markdown(f"✅ **Valid** against `{verdict['checked_against']}`"))
    elif verdict["state"] == "invalid":
        display(Markdown(
            f"❌ **{verdict['error_count']} schema error(s)** - not ready to submit. "
            "Ask the agent to fix these and re-run this cell:\n\n" + "\n".join(
                f"- `{e['path']}` — {e['message']}" for e in verdict["errors"])))
    else:
        display(Markdown(f"⚠️ **Unvalidated** - {verdict.get('note', verdict['state'])}"))

    path = pathlib.Path("/content/cdh_record.yaml")
    path.write_text(record, encoding="utf-8")
    print(f"\nSaved to {path}")
    from google.colab import files
    files.download(str(path))
else:
    print("\nNo fenced YAML found - copy it from the output above.")

---
## What next?

- **Review before you publish.** Any value the agent proposed is its recollection of a public dataset,
  not a fact read from your file. Check `description`, `keywords` and the variable definitions.
- **Submit** the record to the [CDH catalogue](https://github.com/CGIAR-Climate-Data-Hub/cdh-catalog).
- **Read the standard**: [CDH metadata standard v0.3.0](https://github.com/CGIAR-Climate-Data-Hub/cdh-metadata-standard)
  · [authoring guide](https://github.com/CGIAR-Climate-Data-Hub/cdh-metadata-standard/blob/main/spec/authoring-guide.md)
- **More skills**: [CGIAR-Climate-Data-Hub/skills](https://github.com/CGIAR-Climate-Data-Hub/skills)